In [ ]:
# ============================================================
# Block A: 最终修复版 (cftime 硬核偏移 + 物理 1-103 接 002 偏移)
# ============================================================
import os
import glob
import numpy as np
import xarray as xr
from tqdm.auto import tqdm

# --- 配置区域 ---
DATA_DIR_B2000 = "/mnt/soclim0/public_data/weiji/B2000WCN001002/O3"
DATA_DIR_BWCN = "/mnt/soclim0/public_data/weiji/BWCN/O3"
OUT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

PRESSURE_RANGES = [(30, 70, "30_70hPa"), (1, 100, "1_100hPa")]

def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """混合层 O3 柱含量积分逻辑"""
    g, M_air, Na = 9.80665, 28.964 / 1000.0, 6.02214e23
    factor = Na / (g * M_air * 2.687e20)
    P0, PS = ds_sub['P0'], ds_sub['PS']
    P_interface = ds_sub['hyai'] * P0 + ds_sub['hybi'] * PS
    p_i = P_interface.isel(ilev=slice(0, -1)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({'ilev': 'lev'}).assign_coords(lev=ds_sub['lev'])
    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)
    pT, pB = p_top_hpa * 100.0, p_bot_hpa * 100.0
    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)
    return (ds_sub['O3'] * overlap * factor).sum(dim='lev')

def process_b2000wcn():
    exp_out_dir = os.path.join(OUT_ROOT, "B2000WCN")
    os.makedirs(exp_out_dir, exist_ok=True)
    
    all_files = sorted(glob.glob(os.path.join(DATA_DIR_B2000, "B2000WCN.sample.cam.h3.*.O3.nc")))
    
    # 1. 物理筛选：1-103 (Run 001) 和 105-210 (Run 002)
    # 物理文件名 104 的那个残缺文件被明确跳过
    files_001 = [f for f in all_files if 1 <= int(os.path.basename(f).split('.')[-3]) <= 103]
    files_002 = [f for f in all_files if 105 <= int(os.path.basename(f).split('.')[-3]) <= 210]
    
    print(f"[B2000WCN] Loading Run 001 (Y1-103)...")
    ds1 = xr.open_mfdataset(files_001, combine='by_coords')
    
    print(f"[B2000WCN] Loading Run 002 (Y105-210) and shifting internally to Y104-Y209...")
    ds2 = xr.open_mfdataset(files_002, combine='by_coords')
    
    # --- 修正 cftime 时间轴：内部 Y1 -> 物理 Y104 ---
    offset = 103
    old_times = ds2.time.values
    # 直接对 cftime 对象进行年份替换，不通过 pandas
    new_times = [t.replace(year=t.year + offset) for t in old_times]
    ds2 = ds2.assign_coords(time=new_times)
    
    # 2. 合并：此时 ds1 到 103 年，ds2 从 104 年开始
    ds = xr.concat([ds1, ds2], dim='time').sortby('time')
    ds_polar = ds.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(ds_polar.lat))

    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"  Calculating {tag}...")
        # compute() 确保在写入前解决所有 Dask 延迟计算和文件句柄依赖
        o3_ts = calc_parc_o3_hybrid(ds_polar, ptop, pbot).weighted(weights).mean(dim=["lat", "lon"]).compute()
        
        out_nc = os.path.join(exp_out_dir, f"O3_pc_B2000WCN_{tag}.nc")
        o3_ts.to_netcdf(out_nc)
        
        # --- 5-day running mean (Friedel-style) ---
        # min_periods=5：避免边界处不完整窗口参与极值
        o3_5d = o3_ts.rolling(time=5, center=True, min_periods=5).mean()

        # 只在 3-4月找 minimum（用 5d 平均后的日序列）
        spring_all = o3_5d.sel(time=o3_5d.time.dt.month.isin([3, 4]))

        # 有效年份：3-4月至少 ~58 个“有效5日窗口日”（你原阈值保留即可）
        day_counts = spring_all.groupby("time.year").count()
        valid_yrs = day_counts.year.where(day_counts >= 58, drop=True)

        # 5-day mean 的 minimum
        spring_min = spring_all.groupby("time.year").min().sel(year=valid_yrs)
        
        # 保存排序 TXT
        idx_sort = np.argsort(spring_min.values)
        np.savetxt(os.path.join(exp_out_dir, f"lowest10_years_sorted_{tag}.txt"), 
                   spring_min.year.values[idx_sort[:10]], fmt="%04d")
        np.savetxt(os.path.join(exp_out_dir, f"low25pct_years_{tag}.txt"), 
                   spring_min.year.values[idx_sort[:max(1, int(0.25*len(spring_min)))]], fmt="%04d")

def process_bwcn_simple():
    exp_out_dir = os.path.join(OUT_ROOT, "BWCN")
    os.makedirs(exp_out_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(DATA_DIR_BWCN, "BWCN.cam.h3.*.O3.nc")))
    ds = xr.open_mfdataset(files, combine='by_coords').sortby('time')
    ds_polar = ds.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(ds_polar.lat))

    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"[BWCN] Calculating {tag}...")
        o3_ts = calc_parc_o3_hybrid(ds_polar, ptop, pbot).weighted(weights).mean(dim=["lat", "lon"]).compute()
        o3_ts.to_netcdf(os.path.join(exp_out_dir, f"O3_pc_BWCN_{tag}.nc"))
        
        spring_all = o3_ts.sel(time=o3_ts.time.dt.month.isin([3, 4]))
        day_counts = spring_all.groupby("time.year").count()
        valid_yrs = day_counts.year.where(day_counts >= 58, drop=True)
        spring_min = spring_all.groupby("time.year").min().sel(year=valid_yrs)
        idx_sort = np.argsort(spring_min.values)
        np.savetxt(os.path.join(exp_out_dir, f"lowest10_years_sorted_{tag}.txt"), spring_min.year.values[idx_sort[:10]], fmt="%04d")
        np.savetxt(os.path.join(exp_out_dir, f"low25pct_years_{tag}.txt"), spring_min.year.values[idx_sort[:max(1, int(0.25*len(spring_min)))]], fmt="%04d")

if __name__ == "__main__":
    process_b2000wcn()
    process_bwcn_simple()
    print("\n[Block A] All Done. Run 002 (internally Y1) is now physical Y104.")

In [ ]:
# ============================================================
# Block B (O3): 极区臭氧柱折线图 (18:4 比例 & 边缘线增强版)
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import rcParams

# --- Nature Style 细节配置 ---
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.linewidth": 1.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TOTAL_DAYS = 365
N_PREV = 92  
EXP_NAMES = ["B2000WCN", "BWCN"]

# 路径配置
DATA_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

PRESSURE_RANGES = [(30, 70, "30_70hPa"), (1, 100, "1_100hPa")]

def get_shifted_lines(da, years_list, days_needed=365):
    """跨年拼接：Oct(Y-1) 到 Sep(Y)"""
    collected_lines, valid_years = [], []
    ts = da.to_series()
    for y in years_list:
        try:
            seg = pd.concat([ts[f"{int(y-1):04d}-10-01":f"{int(y-1):04d}-12-31"],
                             ts[f"{int(y):04d}-01-01":f"{int(y):04d}-09-30"]])
            if len(seg) >= days_needed:
                collected_lines.append(seg.values[:days_needed])
                valid_years.append(y)
        except: continue
    return np.array(collected_lines), np.array(valid_years)

# --- 精准颜色匹配 (tab10 索引 0, 3, 6, 9 + 新增绿色索引 2) ---
colors_spec = ['#1f77b4', '#d62728', '#e377c2', '#17becf', '#2ca02c']

for name in EXP_NAMES:
    # 建立输出目录：result/plots/[实验名]/O3
    plot_save_dir = os.path.join(PLOT_ROOT, name, "O3")
    os.makedirs(plot_save_dir, exist_ok=True)
    
    print(f"Generating matched-color O3 plots for {name}...")
    
    for ptop, pbot, tag in PRESSURE_RANGES:
        nc_file = os.path.join(DATA_ROOT, name, f"O3_pc_{name}_{tag}.nc")
        low10_file = os.path.join(DATA_ROOT, name, f"lowest10_years_sorted_{tag}.txt")
        low25_file = os.path.join(DATA_ROOT, name, f"low25pct_years_{tag}.txt")
        
        if not os.path.exists(nc_file): continue
        
        da = xr.open_dataarray(nc_file)
        low10_sorted = np.loadtxt(low10_file, dtype=int)
        low25_list = np.loadtxt(low25_file, dtype=int)
        all_years = np.unique(da.time.dt.year.values)
        
        clim_lines, _ = get_shifted_lines(da, all_years[1:])
        clim_mean, clim_std = np.nanmean(clim_lines, axis=0), np.nanstd(clim_lines, axis=0)
        low25_lines, _ = get_shifted_lines(da, low25_list)
        low25_mean, low25_std = np.nanmean(low25_lines, axis=0), np.nanstd(low25_lines, axis=0)
        low5_lines, low5_yrs = get_shifted_lines(da, low10_sorted[:5])
        
        # --- 核心修改：figsize 设定为 (12, 4) ---
        fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
        x = np.arange(TOTAL_DAYS)
        
        # --- 阴影区 A: 气候态 (深灰 + 实心边缘线) ---
        ax.fill_between(x, clim_mean-clim_std, clim_mean+clim_std, color="0.75", alpha=1.0, linewidth=0, zorder=0)
        ax.plot(x, clim_mean-clim_std, color="0.4", lw=0.8, ls="-", label="_nolegend_", zorder=1)
        ax.plot(x, clim_mean+clim_std, color="0.4", lw=0.8, ls="-", label="_nolegend_", zorder=1)
        ax.plot(x, clim_mean, ls="--", lw=1.2, color="0.3", zorder=2)
        
        # --- 阴影区 B: Low 25% (斜线 + 黑色虚线边缘) ---
        ax.fill_between(x, low25_mean-low25_std, low25_mean+low25_std, facecolor="none", edgecolor="0.4", hatch="////", linewidth=0.0, zorder=1)
        ax.plot(x, low25_mean-low25_std, color="black", lw=0.8, ls="--", alpha=0.6, label="_nolegend_", zorder=2)
        ax.plot(x, low25_mean+low25_std, color="black", lw=0.8, ls="--", alpha=0.6, label="_nolegend_", zorder=2)
        ax.plot(x, low25_mean, ls="-", lw=1.8, color="black", zorder=3)
        
        # 极端年曲线
        for i in range(len(low5_lines)):
            ax.plot(x, low5_lines[i], lw=2.2, color=colors_spec[i], label=f"Year {int(low5_yrs[i]):04d}", zorder=10)
            
        # 4. 固定 Y 轴
        if tag == "30_70hPa":
            ax.set_ylim(70, 130)
        elif tag == "1_100hPa":
            ax.set_ylim(200, 350)
            
        # 5. 装饰
        ax.axvline(x=N_PREV, color="k", ls="-", lw=1.0, alpha=0.3)
        ax.set_xlim(0, TOTAL_DAYS)
        ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 242, 273, 304, 334])
        ax.set_xticklabels(["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"])
        ax.set_ylabel("O3 Column (DU)")
        ax.set_title(f"{name} | 60-90°N Polar Ozone Evolution | {ptop}–{pbot} hPa", loc='left', fontweight='bold', fontsize=12)
        
        # 6. 图例重排：下方横向排布
        patch_all = Patch(facecolor="0.75", edgecolor="0.4", label="All-year ±1$\sigma$")
        patch_low = Patch(facecolor="none", edgecolor="0.4", hatch="////", label="Low-ozone ±1$\sigma$")
        line_all  = Line2D([0], [0], color="0.3", lw=1.2, ls="--", label="Climatology Mean")
        line_low  = Line2D([0], [0], color="black", lw=1.8, ls="-", label="Low-ozone Composite")
        handles_years = [Line2D([0], [0], color=colors_spec[i], lw=2.2, label=f"Year {int(low5_yrs[i]):04d}") for i in range(len(low5_lines))]
        
        ax.legend(handles=handles_years + [line_all, patch_all, line_low, patch_low], 
                  loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False, fontsize=9, ncol=5)
        
        # 7. 保存
        plt.savefig(os.path.join(plot_save_dir, f"{name}_O3_Long_Ratio_{tag}.png"), dpi=300, bbox_inches='tight')
        plt.show()

print("Block B (O3) complete. Plots are in 'result/plots/[Exp]/O3'.")

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
from tqdm.auto import tqdm

# --- 配置区域 ---
RAW_DATA_ROOT = "/mnt/soclim0/public_data/weiji"
SAVE_BASE_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

# 数据配置
DATA_CONFIGS = {
    "B2000WCN": {
        "t_dir": os.path.join(RAW_DATA_ROOT, "B2000WCN001002/T"),
        "pattern": "B2000WCN.sample.cam.h3.*.T.nc",
    },
    "BWCN": {
        "t_dir": os.path.join(RAW_DATA_ROOT, "BWCN/T"),
        "pattern": "BWCN.cam.h3.*.T.nc",
    }
}

def calc_minT_pcap(ds, var="T"):
    """计算 60-90°N 极区 zonal mean 的最低温度 (Minimum T)"""
    da = ds[var]
    
    # 1. 经度平均 (Zonal Mean)
    if "lon" in da.dims:
        da = da.mean(dim="lon")
    
    # 2. 选取 60-90N 纬度带并求纬度上的最小值
    da_cap = da.sel(lat=slice(60, 90))
    da_min = da_cap.min(dim="lat")  # 结果维度: (time, lev)

    # 3. 统一垂直坐标维名称为 'lev' 并统一单位为 Pa
    lev_name = None
    for name in ("lev", "plev", "lev_p", "level"):
        if name in da_min.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(f"No vertical dim found in {da_min.dims}")

    lev_vals = da_min[lev_name].values
    max_val = float(np.nanmax(lev_vals))

    # 如果最大值较小（如 1000 左右），判定单位为 hPa，需转为 Pa
    if max_val <= 2000.0:
        lev_pa = lev_vals * 100.0
    else:
        lev_pa = lev_vals

    da_min = da_min.rename({lev_name: "lev"})
    da_min = da_min.assign_coords(lev=lev_pa)
    return da_min

def process_b2000wcn_t():
    """专门处理 B2000WCN 的 T 数据，包含年份修复逻辑"""
    cfg = DATA_CONFIGS["B2000WCN"]
    out_dir = os.path.join(SAVE_BASE_DIR, "B2000WCN")
    os.makedirs(out_dir, exist_ok=True)
    
    all_files = sorted(glob.glob(os.path.join(cfg['t_dir'], cfg['pattern'])))
    
    # 物理筛选：1-103 (Run 001) 和 105-210 (Run 002)
    # 根据文件名中的年份数字进行切分
    files_001 = [f for f in all_files if 1 <= int(os.path.basename(f).split('.')[-3]) <= 103]
    files_002 = [f for f in all_files if 105 <= int(os.path.basename(f).split('.')[-3]) <= 210]
    
    print(f"[B2000WCN T] Loading Run 001 (Y1-103)...")
    ds1 = xr.open_mfdataset(files_001, combine='by_coords', chunks={'time': 365, 'lat': 48})
    
    print(f"[B2000WCN T] Loading Run 002 (Y105-210) and shifting to Y104-209...")
    ds2 = xr.open_mfdataset(files_002, combine='by_coords', chunks={'time': 365, 'lat': 48})
    
    # --- 核心修复：cftime 时间轴偏移 ---
    # Run 002 内部时间从 0001 开始，物理上它是接在 103 年后的第 104 年开始
    offset = 103
    new_times = [t.replace(year=t.year + offset) for t in ds2.time.values]
    ds2 = ds2.assign_coords(time=new_times)
    
    # 合并
    ds = xr.concat([ds1, ds2], dim='time').sortby('time')
    
    print(f"[B2000WCN T] Calculating Min T...")
    t_min_da = calc_minT_pcap(ds, var="T").compute() # .compute() 释放 Dask 句柄
    
    # 属性与保存
    t_min_da.name = "T_min_60_90N"
    t_min_da.attrs["units"] = "K"
    t_min_da["lev"].attrs["units"] = "Pa"
    
    out_file = os.path.join(out_dir, "T_min_60_90N_B2000WCN.nc")
    t_min_da.to_netcdf(out_file)
    print(f"✅ Success: B2000WCN (T_min saved to {out_file})")

def process_bwcn_t_simple():
    """处理普通的 BWCN T 数据"""
    cfg = DATA_CONFIGS["BWCN"]
    out_dir = os.path.join(SAVE_BASE_DIR, "BWCN")
    os.makedirs(out_dir, exist_ok=True)
    
    files = sorted(glob.glob(os.path.join(cfg['t_dir'], cfg['pattern'])))
    ds = xr.open_mfdataset(files, combine='by_coords', chunks={'time': 365, 'lat': 48}).sortby('time')
    
    print(f"[BWCN T] Calculating Min T...")
    t_min_da = calc_minT_pcap(ds, var="T").compute()
    
    t_min_da.name = "T_min_60_90N"
    t_min_da.attrs["units"] = "K"
    t_min_da["lev"].attrs["units"] = "Pa"
    
    out_file = os.path.join(out_dir, "T_min_60_90N_BWCN.nc")
    t_min_da.to_netcdf(out_file)
    print(f"✅ Success: BWCN (T_min saved to {out_file})")

if __name__ == "__main__":
    # 执行修复版的处理函数
    process_b2000wcn_t()
    process_bwcn_t_simple()
    print(f"\n[Block A (T)] All Completed. Data saved in {SAVE_BASE_DIR}")

In [ ]:
# ============================================================
# Block B (T): 极区最低温度折线图 (18:4 比例 & 边缘线增强版)
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import rcParams

# --- Nature Style 细节配置 ---
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.linewidth": 1.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TOTAL_DAYS = 365
N_PREV = 92  
EXP_NAMES = ["B2000WCN", "BWCN"]
DATA_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

TARGET_LEVELS_HPA = [10.0, 50.0, 100.0]
colors_spec = ['#1f77b4', '#d62728', '#e377c2', '#17becf', '#2ca02c']

def get_shifted_lines(da, years_list, days_needed=365):
    collected_lines, valid_years = [], []
    ts = da.to_series()
    for y in years_list:
        try:
            seg = pd.concat([ts[f"{int(y-1):04d}-10-01":f"{int(y-1):04d}-12-31"],
                             ts[f"{int(y):04d}-01-01":f"{int(y):04d}-09-30"]])
            if len(seg) >= days_needed:
                collected_lines.append(seg.values[:days_needed])
                valid_years.append(y)
        except: continue
    return np.array(collected_lines), np.array(valid_years)

def get_level_index(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)

for name in EXP_NAMES:
    plot_save_dir = os.path.join(PLOT_ROOT, name, "T")
    os.makedirs(plot_save_dir, exist_ok=True)
    
    t_nc_file = os.path.join(DATA_ROOT, name, f"T_min_60_90N_{name}.nc")
    if not os.path.exists(t_nc_file): continue
    
    t_min_da_all = xr.open_dataarray(t_nc_file)
    lev_vals = t_min_da_all["lev"].values
    
    low10_file = os.path.join(DATA_ROOT, name, "lowest10_years_sorted_30_70hPa.txt")
    low25_file = os.path.join(DATA_ROOT, name, "low25pct_years_30_70hPa.txt")
    low10_sorted = np.loadtxt(low10_file, dtype=int)
    low25_list = np.loadtxt(low25_file, dtype=int)
    all_years = np.unique(t_min_da_all.time.dt.year.values)

    for target_hpa in TARGET_LEVELS_HPA:
        idx, lev_actual_hpa = get_level_index(lev_vals, target_hpa)
        t_min_da = t_min_da_all.isel(lev=idx)
        
        clim_lines, _ = get_shifted_lines(t_min_da, all_years[1:])
        clim_mean, clim_std = np.nanmean(clim_lines, axis=0), np.nanstd(clim_lines, axis=0)
        low25_lines, _ = get_shifted_lines(t_min_da, low25_list)
        low25_mean, low25_std = np.nanmean(low25_lines, axis=0), np.nanstd(low25_lines, axis=0)
        low5_lines, low5_yrs = get_shifted_lines(t_min_da, low10_sorted[:5])
        
        # --- 尺寸设定 (18:4) ---
        fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
        x = np.arange(TOTAL_DAYS)
        
        # --- 阴影区 A: 气候态 (深灰 + 实心边缘) ---
        ax.fill_between(x, clim_mean-clim_std, clim_mean+clim_std, color="0.75", alpha=1.0, linewidth=0, zorder=0)
        ax.plot(x, clim_mean-clim_std, color="0.4", lw=0.8, ls="-", label="_nolegend_", zorder=1)
        ax.plot(x, clim_mean+clim_std, color="0.4", lw=0.8, ls="-", label="_nolegend_", zorder=1)
        ax.plot(x, clim_mean, ls="--", lw=1.2, color="0.3", zorder=2)
        
        # --- 阴影区 B: Low 25% (斜线 + 黑色虚线边缘) ---
        ax.fill_between(x, low25_mean-low25_std, low25_mean+low25_std, facecolor="none", edgecolor="0.4", hatch="////", linewidth=0.0, zorder=1)
        ax.plot(x, low25_mean-low25_std, color="black", lw=0.8, ls="--", alpha=0.6, label="_nolegend_", zorder=2)
        ax.plot(x, low25_mean+low25_std, color="black", lw=0.8, ls="--", alpha=0.6, label="_nolegend_", zorder=2)
        ax.plot(x, low25_mean, ls="-", lw=1.8, color="black", zorder=3)
        
        # 极端年曲线
        for i in range(len(low5_lines)):
            ax.plot(x, low5_lines[i], lw=2.2, color=colors_spec[i], label=f"Year {int(low5_yrs[i]):04d}", zorder=10)
            
        # 氯活化阈值线
        ax.axhline(y=197, color='royalblue', linestyle='--', linewidth=1.2, zorder=5)
        ax.text(5, 185, 'Cl activation threshold (197K)', color='royalblue', fontsize=10, fontweight='bold')
        
        # 固定轴范围
        ax.set_ylim(180, 250)
        ax.axvline(x=N_PREV, color="k", ls="-", lw=1.0, alpha=0.3)
        ax.set_xlim(0, TOTAL_DAYS)
        ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 242, 273, 304, 334])
        ax.set_xticklabels(["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"])
        
        ax.set_ylabel("Min Temp (K)")
        ax.set_title(f"{name} | 60-90°N Min Temperature | ~{int(round(lev_actual_hpa))} hPa", loc='left', fontweight='bold', fontsize=12)
        
        # 图例重排
        patch_all = Patch(facecolor="0.75", edgecolor="0.4", label="All-year ±1$\sigma$")
        patch_low = Patch(facecolor="none", edgecolor="0.4", hatch="////", label="Low-ozone ±1$\sigma$")
        line_all  = Line2D([0], [0], color="0.3", lw=1.2, ls="--", label="Climatology Mean")
        line_low  = Line2D([0], [0], color="black", lw=1.8, ls="-", label="Low-ozone Composite")
        handles_years = [Line2D([0], [0], color=colors_spec[i], lw=2.2, label=f"Year {int(low5_yrs[i]):04d}") for i in range(len(low5_lines))]
        
        ax.legend(handles=handles_years + [line_all, patch_all, line_low, patch_low], 
                  loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False, fontsize=9, ncol=5)
        
        plt.savefig(os.path.join(plot_save_dir, f"{name}_Tmin_Long_Ratio_{int(round(lev_actual_hpa))}hPa.png"), dpi=300, bbox_inches='tight')
        plt.show()

print("Block B (T) complete. Plots are in 'result/plots/[Exp]/T'.")

In [ ]:
# ============================================================
# Block A (U): 最终修复版 - 物理时间轴对齐与 60N 纬向风提取
# ============================================================
import os
import glob
import numpy as np
import xarray as xr
from tqdm.auto import tqdm
from dask.diagnostics import ProgressBar

# --- 配置区域 ---
RAW_DATA_ROOT = "/mnt/soclim0/public_data/weiji"
SAVE_BASE_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

DATA_CONFIGS = {
    "B2000WCN": {
        "u_dir": os.path.join(RAW_DATA_ROOT, "B2000WCN001002/U"),
        "pattern": "B2000WCN.sample.cam.h3.*.U.nc",
        "out_dir": os.path.join(SAVE_BASE_DIR, "B2000WCN")
    },
    "BWCN": {
        "u_dir": os.path.join(RAW_DATA_ROOT, "BWCN/U"),
        "pattern": "BWCN.cam.h3.*.U.nc",
        "out_dir": os.path.join(SAVE_BASE_DIR, "BWCN")
    }
}

def calc_U60N(ds, var="U"):
    """提取 60°N 附近的纬向平均风 U(time, lev)"""
    da = ds[var]
    
    # 1. 经度平均 (Zonal Mean)
    if "lon" in da.dims:
        da = da.mean(dim="lon")
    
    # 2. 选取 60N (使用 method="nearest" 兼容不同网格)
    if "lat" not in da.dims:
        raise ValueError(f"{var} 缺少 lat 维度")
    da_60N = da.sel(lat=60.0, method="nearest")

    # 3. 识别垂直维并统一压力单位为 Pa
    lev_name = next((n for n in ("lev", "plev", "lev_p", "level") if n in da_60N.dims), None)
    if lev_name is None: 
        raise ValueError(f"无法识别垂直坐标维")

    lev_vals = da_60N[lev_name].values
    max_v = float(np.nanmax(lev_vals))
    # 若最大值小于 2000，判定为 hPa，需转为 Pa
    lev_pa = lev_vals * 100.0 if max_v <= 2000.0 else lev_vals
    
    da_60N = da_60N.rename({lev_name: "lev"}).assign_coords(lev=lev_pa)
    return da_60N

def process_u_exp(name):
    cfg = DATA_CONFIGS[name]
    os.makedirs(cfg['out_dir'], exist_ok=True)
    
    search_path = os.path.join(cfg['u_dir'], cfg['pattern'])
    all_files = sorted(glob.glob(search_path))
    
    if not all_files:
        print(f"❌ Error: {name} 未找到文件")
        return

    print(f"\n[{name}] Step 1: 加载并对齐物理年份...")
    
    if name == "B2000WCN":
        # 物理筛选：Run 001 (Y1-103) 和 Run 002 (Y105-210)
        f_001 = [f for f in all_files if 1 <= int(os.path.basename(f).split('.')[-3]) <= 103]
        f_002 = [f for f in all_files if 105 <= int(os.path.basename(f).split('.')[-3]) <= 210]
        
        # 加载两部分
        ds1 = xr.open_mfdataset(f_001, combine='by_coords', chunks={'time': 365, 'lat': 48})
        ds2 = xr.open_mfdataset(f_002, combine='by_coords', chunks={'time': 365, 'lat': 48})
        
        # cftime 偏移：Run 002 内部起始是 Y1，物理对应 Y104 (103 offset)
        offset = 103
        new_times = [t.replace(year=t.year + offset) for t in ds2.time.values]
        ds2 = ds2.assign_coords(time=new_times)
        
        # 物理拼接
        ds = xr.concat([ds1, ds2], dim='time').sortby('time')
    else:
        # BWCN 正常加载
        ds = xr.open_mfdataset(all_files, combine='by_coords', chunks={'time': 365, 'lat': 48}).sortby('time')

    print(f"[{name}] Step 2: 计算 60N Zonal Mean (计算中...)")
    u_op = calc_U60N(ds, var="U")
    
    # 使用 ProgressBar 实时监控计算进度
    with ProgressBar():
        u_60N_da = u_op.compute()
    
    # 属性整理
    u_60N_da.name = "U_60N"
    u_60N_da.attrs["units"] = "m/s"
    u_60N_da["lev"].attrs["units"] = "Pa"
    
    # 保存
    out_file = os.path.join(cfg['out_dir'], f"U_60N_{name}.nc")
    u_60N_da.to_netcdf(out_file)
    
    # 释放资源
    ds.close()
    print(f"✅ [{name}] 处理完成! 结果：{out_file}")

if __name__ == "__main__":
    # 串行执行实验，内部使用 Dask 并行计算
    for exp_name in DATA_CONFIGS.keys():
        process_u_exp(exp_name)
        
    print(f"\n[Block A (U)] 所有实验处理完毕。")

In [ ]:
# ============================================================
# Block B (U): 60°N 纬向风折线图 (18:4 比例 & 边缘线增强版)
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import rcParams

# --- Nature Style 细节配置 ---
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.linewidth": 1.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TOTAL_DAYS = 365
N_PREV = 92  
EXP_NAMES = ["B2000WCN", "BWCN"]
DATA_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

TARGET_LEVELS_HPA = [1.0, 10.0, 50.0, 100.0]
colors_spec = ['#1f77b4', '#d62728', '#e377c2', '#17becf', '#2ca02c']

def get_shifted_lines(da, years_list, days_needed=365):
    collected_lines, valid_years = [], []
    ts = da.to_series()
    for y in years_list:
        try:
            seg = pd.concat([ts[f"{int(y-1):04d}-10-01":f"{int(y-1):04d}-12-31"],
                             ts[f"{int(y):04d}-01-01":f"{int(y):04d}-09-30"]])
            if len(seg) >= days_needed:
                collected_lines.append(seg.values[:days_needed])
                valid_years.append(y)
        except: continue
    return np.array(collected_lines), np.array(valid_years)

def get_level_index(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)

for name in EXP_NAMES:
    plot_save_dir = os.path.join(PLOT_ROOT, name, "U")
    os.makedirs(plot_save_dir, exist_ok=True)
    
    u_nc_file = os.path.join(DATA_ROOT, name, f"U_60N_{name}.nc")
    if not os.path.exists(u_nc_file): continue
    
    u_da_all = xr.open_dataarray(u_nc_file)
    lev_vals = u_da_all["lev"].values
    
    low10_file = os.path.join(DATA_ROOT, name, "lowest10_years_sorted_30_70hPa.txt")
    low25_file = os.path.join(DATA_ROOT, name, "low25pct_years_30_70hPa.txt")
    low10_sorted = np.loadtxt(low10_file, dtype=int)
    low25_list = np.loadtxt(low25_file, dtype=int)
    all_years = np.unique(u_da_all.time.dt.year.values)

    for target_hpa in TARGET_LEVELS_HPA:
        idx, lev_actual_hpa = get_level_index(lev_vals, target_hpa)
        u_da = u_da_all.isel(lev=idx)
        
        clim_lines, _ = get_shifted_lines(u_da, all_years[1:])
        clim_mean, clim_std = np.nanmean(clim_lines, axis=0), np.nanstd(clim_lines, axis=0)
        low25_lines, _ = get_shifted_lines(u_da, low25_list)
        low25_mean, low25_std = np.nanmean(low25_lines, axis=0), np.nanstd(low25_lines, axis=0)
        low5_lines, low5_yrs = get_shifted_lines(u_da, low10_sorted[:5])
        
        # --- 核心修改：figsize 设定为 (12, 4) 以满足比例要求 ---
        fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
        x = np.arange(TOTAL_DAYS)
        
        # --- 阴影区 A: 气候态 ---
        ax.fill_between(x, clim_mean-clim_std, clim_mean+clim_std, color="0.75", alpha=1.0, linewidth=0, zorder=0)
        ax.plot(x, clim_mean-clim_std, color="0.4", lw=0.8, ls="-", label="_nolegend_", zorder=1)
        ax.plot(x, clim_mean+clim_std, color="0.4", lw=0.8, ls="-", label="_nolegend_", zorder=1)
        ax.plot(x, clim_mean, ls="--", lw=1.2, color="0.3", zorder=2)
        
        # --- 阴影区 B: Low 25% (斜线 + 黑色虚线边缘) ---
        ax.fill_between(x, low25_mean-low25_std, low25_mean+low25_std, facecolor="none", edgecolor="0.4", hatch="////", linewidth=0.0, zorder=1)
        ax.plot(x, low25_mean-low25_std, color="black", lw=0.8, ls="--", alpha=0.6, label="_nolegend_", zorder=2)
        ax.plot(x, low25_mean+low25_std, color="black", lw=0.8, ls="--", alpha=0.6, label="_nolegend_", zorder=2)
        ax.plot(x, low25_mean, ls="-", lw=1.8, color="black", zorder=3)
        
        # 极端年曲线
        for i in range(len(low5_lines)):
            ax.plot(x, low5_lines[i], lw=2.2, color=colors_spec[i], label=f"Year {int(low5_yrs[i]):04d}", zorder=10)
            
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.8, alpha=0.5, zorder=5)
        
        # 固定 Y 轴
        if abs(target_hpa - 1.0) < 0.1: ax.set_ylim(-40, 100)
        elif abs(target_hpa - 10.0) < 0.1: ax.set_ylim(-20, 70)
        elif abs(target_hpa - 50.0) < 0.1: ax.set_ylim(-10, 50)
        elif abs(target_hpa - 100.0) < 0.1: ax.set_ylim(-5, 35)
            
        ax.axvline(x=N_PREV, color="k", ls="-", lw=1.0, alpha=0.3)
        ax.set_xlim(0, TOTAL_DAYS)
        ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 242, 273, 304, 334])
        ax.set_xticklabels(["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"])
        ax.set_ylabel("Zonal Wind (m/s)")
        ax.set_title(f"{name} | 60°N Zonal Wind (U) | ~{int(round(lev_actual_hpa))} hPa", loc='left', fontweight='bold', fontsize=12)
        
        # 图例构建：由于图表变窄，图例改为在右侧外边框显示或横向排布
        patch_all = Patch(facecolor="0.75", edgecolor="0.4", label="All-year ±1$\sigma$")
        patch_low = Patch(facecolor="none", edgecolor="0.4", hatch="////", label="Low-ozone ±1$\sigma$")
        line_all  = Line2D([0], [0], color="0.3", lw=1.2, ls="--", label="Climatology Mean")
        line_low  = Line2D([0], [0], color="black", lw=1.8, ls="-", label="Low-ozone Composite")
        handles_years = [Line2D([0], [0], color=colors_spec[i], lw=2.2, label=f"Year {int(low5_yrs[i]):04d}") for i in range(len(low5_lines))]
        
        # 使用更多的 ncol 以适应扁平布局
        ax.legend(handles=handles_years + [line_all, patch_all, line_low, patch_low], 
                  loc="upper center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=9, ncol=5)
        
        plt.savefig(os.path.join(plot_save_dir, f"{name}_U60N_Long_Ratio_{int(round(lev_actual_hpa))}hPa.png"), dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 配置 ----------------
FILE_BWCN = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"

# ✅ 新的 B2000WCN EPflux daily (40–80N mean, time×plev)
FILE_B2000 = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

# ---------------- 工具函数 ----------------
def get_djf_series(da):
    """
    处理 DJF 跨年逻辑并做纬向平均（若有 lat 维）
    """
    # 1) 纬向加权平均 (如果是 3D 数据)
    if 'lat' in da.coords:
        weights = np.cos(np.deg2rad(da.lat))
        da = da.weighted(weights).mean("lat")

    # 2) 季节重采样 (QS-DEC: 12-1-2月一组，标签落在12月)
    seas_mean = da.resample(time='QS-DEC').mean()

    # 3) 提取 DJF (12月标签)
    djf_da = seas_mean.sel(time=seas_mean.time.dt.month == 12)

    # 4) 剔除不完整的首尾年份
    return djf_da.isel(time=slice(1, -1))

def plot_ep_scatter(exp_name, nc_path, txt_path, out_dir):
    print(f"[{exp_name}] Reading data: {nc_path}")
    ds = xr.open_dataset(nc_path)

    # --- 变量名识别：新 B2000 是 Fz；BWCN 是 EP2_cart ---
    if "Fz" in ds:
        var_name = "Fz"
    elif "EP2_cart" in ds:
        var_name = "EP2_cart"
    else:
        raise KeyError(f"[{exp_name}] Cannot find 'Fz' or 'EP2_cart' in {nc_path}")

    # --- 符号修正：向上为正（沿用你原逻辑：乘 -1） ---
    fz_array = -1.0 * ds[var_name]

    # --- 提取层次并做 DJF 平均 ---
    # 10000 Pa = 100 hPa; 30000 Pa = 300 hPa
    fz100 = get_djf_series(fz_array.sel(plev=10000, method='nearest'))
    fz300 = get_djf_series(fz_array.sel(plev=30000, method='nearest'))

    # 确保时间对齐
    fz100, fz300 = xr.align(fz100, fz300)

    # 标准化
    x = (fz300 - fz300.mean()) / fz300.std()
    y = (fz100 - fz100.mean()) / fz100.std()

    # 读取关键年份文件，并只取前三个
    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            key_years = [int(line.strip()) for line in f.readlines() if line.strip()][:3]
    else:
        key_years = []
    print(f"[{exp_name}] Highlighting top 3 years: {key_years}")

    # ---- 绘图 (修改版：彩色球 + 侧边图例) ----
    plt.figure(figsize=(8.5, 7), dpi=120)  # 增加宽度以容纳右侧图例
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.axhline(0, color='k', lw=0.8, alpha=0.5)
    plt.axvline(0, color='k', lw=0.8, alpha=0.5)

    # 1. 准备颜色和年份
    key_colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    # 这里的 plot_years 是 Dec 标签 + 1 后的物理年份
    plot_years = x.time.dt.year.values + 1
    
    # 2. 区分背景年和关键年
    bg_mask = ~np.isin(plot_years, key_years)
    
    # 3. 绘制背景点 (Other Years)
    plt.scatter(x.values[bg_mask], y.values[bg_mask], 
                color='black', alpha=0.3, s=40, edgecolors='none', 
                label='Other Years', zorder=3)

    # 4. 逐个绘制关键年彩点并添加到图例
    for i, yr in enumerate(key_years):
        if yr in plot_years:
            idx = np.where(plot_years == yr)[0][0]
            plt.scatter(x.values[idx], y.values[idx], 
                        color=key_colors[i % len(key_colors)], 
                        s=80, edgecolors='k', linewidths=0.8, 
                        label=f"Year {yr:04d}", zorder=10)

    # 5. 统计量与装饰
    r_val, p_val = pearsonr(x.values, y.values)
    plt.title(f"{exp_name} DJF $F_z$ Correlation (100 vs 300 hPa)", fontsize=13, pad=15)
    plt.xlabel("DJF $F_z$ at 300 hPa (Standardized $\sigma$)", fontsize=11)
    plt.ylabel("DJF $F_z$ at 100 hPa (Standardized $\sigma$)", fontsize=11)
    plt.xlim(-3.5, 3.5)
    plt.ylim(-3.5, 3.5)

    # 6. 信息框
    info_text = f"r = {r_val:.3f}\np = {p_val:.2e}\nN = {len(x)}"
    plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes,
             va='top', ha='left', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'))

    # 7. 图例放置在右侧
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, fontsize=10)
    
    plt.tight_layout() # 自动调整布局防止标签切掉

    # 统计量
    r_val, p_val = pearsonr(x.values, y.values)

    # 装饰
    plt.title(f"{exp_name} DJF $F_z$ Correlation (100 vs 300 hPa)", fontsize=13, pad=15)
    plt.xlabel("DJF $F_z$ at 300 hPa (Standardized $\sigma$)", fontsize=11)
    plt.ylabel("DJF $F_z$ at 100 hPa (Standardized $\sigma$)", fontsize=11)

    plt.xlim(-3, 3)
    plt.ylim(-3, 3)

    info_text = f"r = {r_val:.3f}\np = {p_val:.2e}\nN = {len(x)}"
    plt.text(0.05, 0.95, info_text, transform=plt.gca().transAxes,
             va='top', ha='left', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'))

    os.makedirs(out_dir, exist_ok=True)
    save_name = os.path.join(out_dir, f"EP_Flux_Scatter_{exp_name}_100vs300.png")
    plt.savefig(save_name, bbox_inches='tight', dpi=300)
    print(f"[{exp_name}] Plot saved to: {save_name}")
    plt.show()

    ds.close()

# ---------------- 执行循环 ----------------
tasks = [
    ("BWCN", FILE_BWCN,
     os.path.join(DATA_BASE, "BWCN/lowest10_years_sorted_30_70hPa.txt"),
     os.path.join(PLOT_BASE, "BWCN/EP FLUX散点图")),

    ("B2000WCN", FILE_B2000,
     os.path.join(DATA_BASE, "B2000WCN/lowest10_years_sorted_30_70hPa.txt"),
     os.path.join(PLOT_BASE, "B2000WCN/EP FLUX散点图"))
]

for name, nc, txt, out in tasks:
    if os.path.exists(nc):
        plot_ep_scatter(name, nc, txt, out)
    else:
        print(f"❌ Missing data file for {name}: {nc}")


import xarray as xr, numpy as np
ds = xr.open_dataset("/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc")
t = ds.time.values
print("len(time)=", len(t), " unique=", len(np.unique(t)))
print("min=", t.min(), " max=", t.max())





In [ ]:
# ============================================================
# Block C (Scatter Pro): 可定制时间窗与统计维度的 EP-O3 分析
# 修正版：仅对拼接型实验（如 B2000WCN）在跨年 Fz 季节中剔除连接年 104
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

# 只有这些实验存在 001/002 拼接导致的“连接年”
BRIDGE_EXPERIMENTS = {"B2000WCN", "B2000WCN_NOCOUPL"}
BRIDGE_YEAR = 104

# 月份映射表
MONTH_MAP = {'J':1, 'F':2, 'M':3, 'A':4, 'm':5, 'j':6, 'x':7, 'a':8, 'S':9, 'O':10, 'N':11, 'D':12}

# 处理一些重名月份（如 March/May, June/July），建议直接用具体月份列表，但这里兼容简写：
SHORT_MAP = {
    "DJF": [12, 1, 2],
    "JF": [1, 2],
    "FM": [2, 3],
    "MA": [3, 4],
    "NDJ": [11, 12, 1],
    "JFM": [1, 2, 3],
    "OND": [10, 11, 12]
}

def is_cross_year_season(season_str):
    """判断 season 是否跨年（如 DJF / NDJ）"""
    months = SHORT_MAP.get(season_str, [12, 1, 2])  # 默认 DJF
    return any(m in [11, 12] for m in months) and any(m in [1, 2] for m in months)

def get_seasonal_data(da, season_str, method='mean'):
    """
    处理跨年拼接逻辑并返回按年统计的时间序列
    season_str: 如 "DJF", "NDJ", "MA"
    method: 'mean' 或 'min'
    """
    months = SHORT_MAP.get(season_str, [12, 1, 2])  # 默认 DJF
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    cross_year = is_cross_year_season(season_str)

    for yr in all_years:
        try:
            if cross_year:
                # 拼接逻辑：Year-1 的 late months + Year 的 early months
                early_months = [m for m in months if m < 6]
                late_months = [m for m in months if m >= 6]

                parts = []
                for m in late_months:
                    parts.append(ts[f"{int(yr-1):04d}-{m:02d}"])
                for m in early_months:
                    parts.append(ts[f"{int(yr):04d}-{m:02d}"])
                combined = pd.concat(parts)
            else:
                # 非跨年逻辑：直接提取当年月份
                combined = pd.concat([ts[f"{int(yr):04d}-{m:02d}"] for m in months])

            # 简单检查数据量是否合理 (每个月至少28天)
            if len(combined) < len(months) * 28:
                continue

            val = combined.mean() if method == 'mean' else combined.min()
            results[yr] = val
        except:
            continue  # 如果年份不存在或数据缺失，跳过

    return pd.Series(results).to_xarray().rename({'index': 'year'})

def plot_custom_relationship(exp_name, ep_nc, o3_nc, txt_path,
                             fz_season="DJF", fz_method="mean",
                             o3_season="MA", o3_method="mean"):

    print(f"[{exp_name}] Analyzing Fz({fz_season}, {fz_method}) vs O3({o3_season}, {o3_method})...")

    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        print("  Missing file(s). Skip.")
        return

    # 1. 加载并预处理数据
    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method='nearest')

    if 'lat' in fz_raw.coords:
        fz_raw = fz_raw.weighted(np.cos(np.deg2rad(fz_raw.lat))).mean("lat")

    # 2. 提取指定季节和方法的统计量
    fz_stat = get_seasonal_data(fz_raw, fz_season, method=fz_method)

    # ✅ 关键修正：
    # 仅对“拼接型实验”且 Fz 为跨年季节时，删除连接年 104
    if (exp_name in BRIDGE_EXPERIMENTS) and is_cross_year_season(fz_season):
        if BRIDGE_YEAR in fz_stat.year.values:
            fz_stat = fz_stat.sel(year=fz_stat.year != BRIDGE_YEAR)
            print(f"  Removed bridge year {BRIDGE_YEAR} for {exp_name} cross-year Fz season ({fz_season}).")

    # --- O3: apply 5-day running mean BEFORE seasonal stats ---
    da_o3_5d = da_o3.rolling(time=5, center=True, min_periods=5).mean()
    o3_stat = get_seasonal_data(da_o3_5d, o3_season, method=o3_method)

    # 3. 对齐年份
    common_years = np.intersect1d(fz_stat.year.values, o3_stat.year.values)
    x_raw = fz_stat.sel(year=common_years)
    y_raw = o3_stat.sel(year=common_years)

    if len(common_years) < 3:
        print("  Too few common years after filtering. Skip.")
        ds_ep.close()
        return

    # 4. 标准化
    x = (x_raw - x_raw.mean()) / x_raw.std()
    y = (y_raw - y_raw.mean()) / y_raw.std()

    # 5. 绘图 (修改版：彩色点 + 图例)
    plt.figure(figsize=(7, 6), dpi=120)  # 略微加宽给图例留空间
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.axhline(0, color='k', lw=0.8, alpha=0.5)
    plt.axvline(0, color='k', lw=0.8, alpha=0.5)

    # 获取颜色循环 (Nature style 常用颜色)
    key_colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    key_years = np.loadtxt(txt_path, dtype=int)[:3] if os.path.exists(txt_path) else []

    # 先画背景点
    bg_mask = ~np.isin(x.year.values, np.atleast_1d(key_years))
    plt.scatter(
        x.values[bg_mask], y.values[bg_mask],
        color='black', alpha=0.8, s=40, edgecolors='none',
        label='Other Years'
    )

    # 再画关键年彩点并标记图例
    for i, yr in enumerate(np.atleast_1d(key_years)):
        if yr in x.year.values:
            idx = np.where(x.year.values == yr)[0][0]
            plt.scatter(
                x.values[idx], y.values[idx],
                color=key_colors[i % len(key_colors)],
                s=40, edgecolors='k', linewidths=0.8,
                label=f"Year {int(yr):04d}", zorder=10
            )

    # 图例放在右侧
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)

    r, p = pearsonr(x.values, y.values)
    plt.text(
        0.05, 0.95,
        f"r = {r:.3f}\np = {p:.2e}\nN = {len(x)}",
        transform=plt.gca().transAxes,
        va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none')
    )

    plt.xlim(-3.5, 3.5)
    plt.ylim(-3.5, 3.5)
    plt.xlabel(f"{fz_season} $F_z$ at 100hPa ({fz_method}, $\\sigma$)", fontsize=10)
    plt.ylabel(f"{o3_season} O3 Column ({o3_method}, $\\sigma$)", fontsize=10)
    plt.title(f"{exp_name} | {fz_season} Fz vs {o3_season} O3", loc='left', fontweight='bold')

    out_dir = os.path.join(PLOT_BASE, exp_name, "EP_O3_Custom")
    os.makedirs(out_dir, exist_ok=True)
    save_path = os.path.join(out_dir, f"Scatter_{fz_season}_{fz_method}_vs_{o3_season}_{o3_method}.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print("Saved:", save_path)

    ds_ep.close()

# ---------------- 执行示例 (多配置批量生成) ----------------
# 定义你想要分析的所有配置组合
configs = [
    {
        "fz_season": "DJF",
        "fz_method": "mean",
        "o3_season": "MA",
        "o3_method": "min"
    },
    {
        "fz_season": "JF",
        "fz_method": "mean",
        "o3_season": "MA",
        "o3_method": "min"
    }
]

tasks = [
    (
        "BWCN",
        FILE_BWCN_EP,
        os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN/lowest10_years_sorted_30_70hPa.txt")
    ),
    (
        "B2000WCN",
        FILE_B2000_EP,
        os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN/lowest10_years_sorted_30_70hPa.txt")
    )
]

# 使用嵌套循环：先遍历实验，再遍历配置
for name, ep, o3, txt in tasks:
    for cfg in configs:
        plot_custom_relationship(name, ep, o3, txt, **cfg)

有一个隐藏BUG，链接俩数据的时候 0103这一年的DEC不可以和0104的Jan组合。。但无所谓了。

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# --- 路径配置 ---
DATA_FILE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"
DEBUG_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/B2000WCNDEBUG"
os.makedirs(DEBUG_DIR, exist_ok=True)

# 1. 加载数据并强制排序 (解决 glob 乱序问题)
da = xr.open_dataarray(DATA_FILE).sortby('time')

# 打印一下数据范围，确认 Run 002 是否在 NC 文件里
start_year = da.time.dt.year.min().values
end_year = da.time.dt.year.max().values
print(f">>> 数据文件包含年份范围: {start_year} 到 {end_year}")

# 2. 提取 3-4 月数据并清洗
spring_da = da.sel(time=da.time.dt.month.isin([3, 4]))

# 计算每年在 3-4 月拥有的天数 (正常应为 61 天)
# 注意：有些模式输出可能是 360 天历，则 3-4 月为 60 天
days_per_year = spring_da.groupby("time.year").count()
# 只要天数 > 58 天，我们就认为这个春天是完整的（容忍少量的缺测）
valid_years = days_per_year.year.where(days_per_year >= 58, drop=True).values

print(f">>> 清洗后有效年份数量: {len(valid_years)}")
print(f">>> 有效年份示例: {valid_years[:5]} ... {valid_years[-5:]}")

# 3. 提取最小值并构建 DataFrame
spring_min = spring_da.groupby("time.year").min().sel(year=valid_years)
df = spring_min.to_dataframe(name='O3_min').reset_index()

# 显式标记分组
df['Group'] = 'Unknown'
df.loc[df['year'] <= 104, 'Group'] = 'Run 001'
df.loc[df['year'] > 104, 'Group'] = 'Run 002'

# 检查 Run 002 是否有数据点
n_002 = len(df[df['Group'] == 'Run 002'])
if n_002 == 0:
    print("!!! 警告: Run 002 在清洗后没有任何数据点。请检查原始 NC 文件是否包含 105 年后的数据。")
else:
    print(f">>> Run 002 包含数据点数量: {n_002}")

# 4. 统计分析
g1_data = df[df['Group'] == 'Run 001']['O3_min']
g2_data = df[df['Group'] == 'Run 002']['O3_min']

# --- 绘图 1: Time Series + Boxplot (方案 1 + 2) ---
fig = plt.figure(figsize=(15, 6), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1])

# 子图 A: 时间序列
ax1 = fig.add_subplot(gs[0])
# 绘制所有点
sns.lineplot(data=df, x='year', y='O3_min', hue='Group', marker='o', palette={'Run 001': '#1f77b4', 'Run 002': '#d62728'}, ax=ax1)
# 分段均值虚线
ax1.axhline(g1_data.mean(), color='#1f77b4', ls='--', alpha=0.7, label='001 Mean')
if not g2_data.empty:
    ax1.axhline(g2_data.mean(), color='#d62728', ls='--', alpha=0.7, label='002 Mean')

ax1.axvline(104.5, color='black', lw=1, ls='-')
ax1.set_title("B2000WCN: O3 Min (Mar-Apr) Consistency Check", loc='left', fontsize=14, fontweight='bold')
ax1.set_xlabel("Year of Simulation")
ax1.set_ylabel("O3 Column Minimum (DU)")

# 子图 B: Boxplot
ax2 = fig.add_subplot(gs[1])
sns.boxplot(data=df, x='Group', y='O3_min', palette={'Run 001': '#1f77b4', 'Run 002': '#d62728'}, ax=ax2)
if not g2_data.empty:
    t_stat, p_val = stats.ttest_ind(g1_data, g2_data)
    ax2.set_title(f"Distribution Comparison\nP-value: {p_val:.4e}", fontsize=12)
else:
    ax2.set_title("Distribution Comparison\n(Run 002 Missing)", fontsize=12)

plt.savefig(os.path.join(DEBUG_DIR, "DEBUG_O3_Step_Change.png"), dpi=300)
plt.show()

# --- 绘图 2: PDF (方案 3) ---
if not g2_data.empty:
    plt.figure(figsize=(8, 6))
    sns.kdeplot(data=df, x='O3_min', hue='Group', fill=True, palette={'Run 001': '#1f77b4', 'Run 002': '#d62728'}, alpha=0.4)
    plt.title("O3 Min Probability Density Function: Run 001 vs 002", fontweight='bold')
    plt.xlabel("O3 Column (DU)")
    plt.savefig(os.path.join(DEBUG_DIR, "DEBUG_O3_PDF_Comparison.png"), dpi=300)
    plt.show()

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# 假设这里有你外部定义的 get_seasonal_data 函数
# from your_module import get_seasonal_data

def prep_one_exp_raw(exp, ep_nc, o3_nc, fz_season="DJF", fz_method="mean",
                     o3_season="MA", o3_method="min", apply_o3_5d=True,
                     lat_band=(40, 80), drop_bridge_year=None):  # ✅ 新增参数 drop_bridge_year
    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"

    # ---- 100 hPa daily ----
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    # ✅ 统一：40–80°N 加权平均（仅当存在 lat 维）
    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # yearly stats (raw)
    x_raw = get_seasonal_data(fz_raw, fz_season, method=fz_method)  # DataArray(year)

    # ✅ 新增处理逻辑：丢弃错误的 bridge year（如 B2000WCN 的 104 年）
    if drop_bridge_year is not None:
        if drop_bridge_year in x_raw.year.values:
            x_raw = x_raw.sel(year=(x_raw.year != drop_bridge_year))

    # O3: 5-day running mean then seasonal min/mean
    if apply_o3_5d:
        da_o3 = da_o3.rolling(time=5, center=True, min_periods=5).mean()
    y_raw = get_seasonal_data(da_o3, o3_season, method=o3_method)    # DataArray(year)

    # align years within exp
    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    x_raw = x_raw.sel(year=common_years)
    y_raw = y_raw.sel(year=common_years)

    df = pd.DataFrame({
        "exp": exp,
        "year": x_raw.year.values.astype(int),
        "x_raw": x_raw.values.astype(float),
        "y_raw": y_raw.values.astype(float),
    })

    ds_ep.close()
    return df

def plot_pooled_with_global_top5():
    # -------- paths --------
    FILE_BWCN_EP   = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
    FILE_B2000_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
    DATA_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
    PLOT_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

    o3_bwcn   = os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc")
    o3_b2000  = os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc")

    # -------- config --------
    fz_season, fz_method = "DJF", "mean"
    o3_season, o3_method = "MA",  "min"
    apply_o3_5d = True

    # -------- build pooled table --------
    # ✅ 调用时，针对 B2000WCN 传入 104 年作为需要丢弃的年份
    df_b2000 = prep_one_exp_raw("B2000WCN", FILE_B2000_EP, o3_b2000,
                               fz_season, fz_method, o3_season, o3_method, apply_o3_5d,
                               drop_bridge_year=104) 
    # BWCN 不需要丢弃
    df_bwcn  = prep_one_exp_raw("BWCN", FILE_BWCN_EP, o3_bwcn,
                               fz_season, fz_method, o3_season, o3_method, apply_o3_5d,
                               drop_bridge_year=None)

    df_all = pd.concat([df_b2000, df_bwcn], ignore_index=True).dropna()

    # -------- pooled standardization (μ/σ from both experiments together) --------
    mu_x, sd_x = df_all["x_raw"].mean(), df_all["x_raw"].std(ddof=0)
    mu_y, sd_y = df_all["y_raw"].mean(), df_all["y_raw"].std(ddof=0)

    df_all["x"] = (df_all["x_raw"] - mu_x) / sd_x
    df_all["y"] = (df_all["y_raw"] - mu_y) / sd_y

    # -------- global top 5 extremes: lowest O3 (y_raw smallest) --------
    df_top5 = df_all.sort_values("y_raw", ascending=True).head(5).copy()
    # 给每个极端点一个唯一标签，避免“Year 0008”混淆
    df_top5["label"] = df_top5.apply(lambda r: f"{r['exp']}-{int(r['year']):04d}", axis=1)

    # -------- plot --------
    plt.figure(figsize=(9.2, 6.6), dpi=140)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.axvline(0, color="k", lw=0.8, alpha=0.5)

    # base points
    m_b2000 = df_all["exp"].values == "B2000WCN"
    m_bwcn  = df_all["exp"].values == "BWCN"

    # B2000: gray filled
    plt.scatter(df_all.loc[m_b2000, "x"], df_all.loc[m_b2000, "y"],
                s=45, alpha=0.35, color="0.2", edgecolors="none",
                label="B2000WCN (filled, all years)", zorder=2)

    # BWCN: gray hollow
    plt.scatter(df_all.loc[m_bwcn, "x"], df_all.loc[m_bwcn, "y"],
                s=45, alpha=0.60, facecolors="none", edgecolors="0.2", linewidths=0.8,
                label="BWCN (hollow, all years)", zorder=2)

    # overlay global top5 in colors (顺序：最极端->第5极端)
    colors_spec = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

    for i, (_, r) in enumerate(df_top5.iterrows()):
        c = colors_spec[i]
        if r["exp"] == "B2000WCN":
            # colored filled
            plt.scatter(r["x"], r["y"], s=130, color=c, edgecolors="k", linewidths=0.9,
                        zorder=10, label=f"Top{i+1}: {r['label']}")
        else:
            # colored hollow
            plt.scatter(r["x"], r["y"], s=130, facecolors="none", edgecolors=c, linewidths=2.0,
                        zorder=10, label=f"Top{i+1}: {r['label']}")

    # stats (pooled correlation on standardized values)
    r_all, p_all = pearsonr(df_all["x"].values, df_all["y"].values)

    info = (
        f"POOLED standardization (μ/σ from BWCN + B2000 together)\n"
        f"Global Top5 = lowest O3 (based on raw y before standardization)\n"
        f"ALL: r={r_all:.3f}, p={p_all:.2e}, N={len(df_all)}\n"
        f"O3 = 5-day running mean then {o3_season} {o3_method}"
    )
    plt.text(0.03, 0.97, info, transform=plt.gca().transAxes,
             va="top", ha="left", fontsize=9,
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.75, edgecolor="none"))

    plt.xlabel(f"{fz_season} $F_z$ at 100 hPa (40–80°N mean, pooled standardized)", fontsize=11)
    plt.ylabel(f"{o3_season} O3 ({o3_method} of 5-day mean, pooled standardized)", fontsize=11)
    plt.title("Fz vs Minimum O3 (BWCN + B2000WCN) — Global Top5 highlighted", loc="left", fontweight="bold")

    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
    plt.tight_layout()

    out_dir = os.path.join(PLOT_BASE, "COMBINED")
    os.makedirs(out_dir, exist_ok=True)
    save_path = os.path.join(out_dir, f"Scatter_{fz_season}_{fz_method}_vs_{o3_season}_{o3_method}_O3_5day_GLOBAL_TOP5.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# run
plot_pooled_with_global_top5()

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
def get_seasonal_data(da, season_str, method='mean'):
    """
    处理跨年拼接逻辑并返回按年统计的时间序列
    season_str: 如 "DJF", "NDJ", "MA"
    method: 'mean' 或 'min'
    """
    months = SHORT_MAP.get(season_str, [12, 1, 2])  # 默认 DJF
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    cross_year = is_cross_year_season(season_str)

    for yr in all_years:
        try:
            if cross_year:
                # 拼接逻辑：Year-1 的 late months + Year 的 early months
                early_months = [m for m in months if m < 6]
                late_months = [m for m in months if m >= 6]

                parts = []
                for m in late_months:
                    parts.append(ts[f"{int(yr-1):04d}-{m:02d}"])
                for m in early_months:
                    parts.append(ts[f"{int(yr):04d}-{m:02d}"])
                combined = pd.concat(parts)
            else:
                # 非跨年逻辑：直接提取当年月份
                combined = pd.concat([ts[f"{int(yr):04d}-{m:02d}"] for m in months])

            # 简单检查数据量是否合理 (每个月至少28天)
            if len(combined) < len(months) * 28:
                continue

            val = combined.mean() if method == 'mean' else combined.min()
            results[yr] = val
        except:
            continue  # 如果年份不存在或数据缺失，跳过

    return pd.Series(results).to_xarray().rename({'index': 'year'})

SHORT_MAP = {
    "DJF": [12, 1, 2],
    "JF": [1, 2],
    "FM": [2, 3],
    "MA": [3, 4],
    "NDJ": [11, 12, 1],
    "JFM": [1, 2, 3],
    "OND": [10, 11, 12]
}

# ---------------- 基础配置 (新增用于筛选) ----------------
BRIDGE_EXPERIMENTS = {"B2000WCN", "B2000WCN_NOCOUPL"}
BRIDGE_YEAR = 104

def is_cross_year_season(season_str):
    """判断 season 是否跨年（如 DJF / NDJ）"""
    # 这里的 SHORT_MAP 需与 get_seasonal_data 一致
    s_map = {"DJF": [12, 1, 2], "NDJ": [11, 12, 1], "JF": [1, 2]} 
    months = s_map.get(season_str, [12, 1, 2])
    return any(m in [11, 12] for m in months) and any(m in [1, 2] for m in months)

def prep_one_exp_raw(exp, ep_nc, o3_nc, fz_season="DJF", fz_method="mean",
                     o3_season="MA", o3_method="min", apply_o3_5d=True,
                     lat_band=(40, 80)):
    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"

    # ---- 100 hPa daily ----
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    # ✅ 统一：40–80°N 加权平均
    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # yearly stats (raw)
    x_raw = get_seasonal_data(fz_raw, fz_season, method=fz_method)  # DataArray(year)

    # ✅ 核心修复：仅对拼接型实验在跨年 Fz 季节中剔除连接年 104
    if (exp in BRIDGE_EXPERIMENTS) and is_cross_year_season(fz_season):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=(x_raw.year != BRIDGE_YEAR))
            print(f"  [{exp}] Removed bridge year {BRIDGE_YEAR} for cross-year Fz season ({fz_season}).")

    # O3: 5-day running mean then seasonal min/mean
    if apply_o3_5d:
        da_o3 = da_o3.rolling(time=5, center=True, min_periods=5).mean()
    y_raw = get_seasonal_data(da_o3, o3_season, method=o3_method)    # DataArray(year)

    # align years within exp
    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    x_raw = x_raw.sel(year=common_years)
    y_raw = y_raw.sel(year=common_years)

    df = pd.DataFrame({
        "exp": exp,
        "year": x_raw.year.values.astype(int),
        "x_raw": x_raw.values.astype(float),
        "y_raw": y_raw.values.astype(float),
    })

    ds_ep.close()
    return df

def plot_pooled_with_global_top5():
    # -------- paths --------
    FILE_BWCN_EP   = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
    FILE_B2000_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
    DATA_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
    PLOT_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

    o3_bwcn   = os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc")
    o3_b2000  = os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc")

    # -------- config --------
    fz_season, fz_method = "DJF", "mean"
    o3_season, o3_method = "MA",  "min"
    apply_o3_5d = True

    # -------- build pooled table --------
    df_b2000 = prep_one_exp_raw("B2000WCN", FILE_B2000_EP, o3_b2000,
                               fz_season, fz_method, o3_season, o3_method, apply_o3_5d)
    df_bwcn  = prep_one_exp_raw("BWCN", FILE_BWCN_EP, o3_bwcn,
                               fz_season, fz_method, o3_season, o3_method, apply_o3_5d)

    df_all = pd.concat([df_b2000, df_bwcn], ignore_index=True).dropna()

    # -------- pooled standardization (μ/σ from both experiments together) --------
    mu_x, sd_x = df_all["x_raw"].mean(), df_all["x_raw"].std(ddof=0)
    mu_y, sd_y = df_all["y_raw"].mean(), df_all["y_raw"].std(ddof=0)

    df_all["x"] = (df_all["x_raw"] - mu_x) / sd_x
    df_all["y"] = (df_all["y_raw"] - mu_y) / sd_y

    # -------- global top 5 extremes: lowest O3 (y_raw smallest) --------
    df_top5 = df_all.sort_values("y_raw", ascending=True).head(5).copy()
    # 给每个极端点一个唯一标签，避免“Year 0008”混淆
    df_top5["label"] = df_top5.apply(lambda r: f"{r['exp']}-{int(r['year']):04d}", axis=1)

    # -------- plot --------
    plt.figure(figsize=(9.2, 6.6), dpi=140)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.axvline(0, color="k", lw=0.8, alpha=0.5)

    # base points
    m_b2000 = df_all["exp"].values == "B2000WCN"
    m_bwcn  = df_all["exp"].values == "BWCN"

    # B2000: gray filled
    plt.scatter(df_all.loc[m_b2000, "x"], df_all.loc[m_b2000, "y"],
                s=45, alpha=0.35, color="0.2", edgecolors="none",
                label="B2000WCN (filled, all years)", zorder=2)

    # BWCN: gray hollow
    plt.scatter(df_all.loc[m_bwcn, "x"], df_all.loc[m_bwcn, "y"],
                s=45, alpha=0.60, facecolors="none", edgecolors="0.2", linewidths=0.8,
                label="BWCN (hollow, all years)", zorder=2)

    # overlay global top5 in colors (顺序：最极端->第5极端)
    colors_spec = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

    for i, (_, r) in enumerate(df_top5.iterrows()):
        c = colors_spec[i]
        if r["exp"] == "B2000WCN":
            # colored filled
            plt.scatter(r["x"], r["y"], s=130, color=c, edgecolors="k", linewidths=0.9,
                        zorder=10, label=f"Top{i+1}: {r['label']}")
        else:
            # colored hollow
            plt.scatter(r["x"], r["y"], s=130, facecolors="none", edgecolors=c, linewidths=2.0,
                        zorder=10, label=f"Top{i+1}: {r['label']}")

    # stats (pooled correlation on standardized values)
    r_all, p_all = pearsonr(df_all["x"].values, df_all["y"].values)

    info = (
        f"POOLED standardization (μ/σ from BWCN + B2000 together)\n"
        f"Global Top5 = lowest O3 (based on raw y before standardization)\n"
        f"ALL: r={r_all:.3f}, p={p_all:.2e}, N={len(df_all)}\n"
        f"O3 = 5-day running mean then {o3_season} {o3_method}"
    )
    plt.text(0.03, 0.97, info, transform=plt.gca().transAxes,
             va="top", ha="left", fontsize=9,
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.75, edgecolor="none"))

    plt.xlabel(f"{fz_season} $F_z$ at 100 hPa (40–80°N mean, pooled standardized)", fontsize=11)
    plt.ylabel(f"{o3_season} O3 ({o3_method} of 5-day mean, pooled standardized)", fontsize=11)
    plt.title("Fz vs Minimum O3 (BWCN + B2000WCN) — Global Top5 highlighted", loc="left", fontweight="bold")

    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
    plt.tight_layout()

    out_dir = os.path.join(PLOT_BASE, "COMBINED")
    os.makedirs(out_dir, exist_ok=True)
    save_path = os.path.join(out_dir, f"Scatter_{fz_season}_{fz_method}_vs_{o3_season}_{o3_method}_O3_5day_GLOBAL_TOP5.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# 确保 get_seasonal_data 已定义并运行
plot_pooled_with_global_top5()

In [ ]:
# ============================================================
# Block D: Sliding-window correlation curves (ALL years)
# 目标：看 90-day / 60-day Fz windows 与 MA O3 minimum 的相关系数随窗口起点变化
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_EXPERIMENTS = {"B2000WCN", "B2000WCN_NOCOUPL"}
BRIDGE_YEAR = 104

WINDOW_DAYS_LIST = [90, 60]

def build_window_starts():
    """
    窗口起点：从 12-01 到 02-01，逐日滑动
    """
    starts = []
    # Dec 1-31
    for d in range(1, 32):
        starts.append((12, d))
    # Jan 1-31
    for d in range(1, 32):
        starts.append((1, d))
    # Feb 1
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    返回按 year 的 MA minimum O3（目标变量）
    """
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            # 与你原逻辑一致：至少 ~58 天
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    """
    针对每个 target year，取一个固定长度滑动窗口的 Fz 均值
    - 如果起点在 12 月，则窗口从 (year-1)-12-dd 开始
    - 如果起点在 1/2 月，则窗口从 year-mm-dd 开始
    - 用“从起点开始取前 window_days 个日值”的方式，避免 cftime 日期加法麻烦
    """
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def compute_r_curve(exp_name, ep_nc, o3_nc, lat_band=(40, 80), apply_o3_5d=True, min_n=5):
    """
    返回：
    - labels: 窗口起始日期标签
    - r_curves: dict {window_days: [r1, r2, ...]}
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        print(f"[{exp_name}] Missing file(s).")
        return None, None

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # Fz daily @100 hPa
    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # O3 target: MA minimum of 5-day running mean
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_curves = {wd: [] for wd in WINDOW_DAYS_LIST}

    for window_days in WINDOW_DAYS_LIST:
        for start_month, start_day in starts:
            x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)

            # 连接年修正（仅拼接型实验）
            # 只有当窗口起点在 12 月时，target year=104 才会跨 run 混合
            if (exp_name in BRIDGE_EXPERIMENTS) and (start_month == 12):
                if BRIDGE_YEAR in x_raw.year.values:
                    x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

            common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
            x_use = x_raw.sel(year=common_years)
            y_use = y_raw.sel(year=common_years)

            if len(common_years) < min_n:
                r_curves[window_days].append(np.nan)
                continue

            try:
                r, _ = pearsonr(x_use.values, y_use.values)
            except:
                r = np.nan

            r_curves[window_days].append(r)

    ds_ep.close()
    return labels, r_curves

def plot_r_curves(exp_name, labels, r_curves, suffix="ALL"):
    out_dir = os.path.join(PLOT_BASE, exp_name, "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))

    plt.plot(x, r_curves[90], lw=2.0, marker="o", ms=3, label="90-day window")
    plt.plot(x, r_curves[60], lw=2.0, marker="o", ms=3, label="60-day window")

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels)-1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(f"{exp_name} | Sliding-window correlation with MA minimum O3 ({suffix})",
              loc="left", fontweight="bold")

    plt.legend(frameon=False)
    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_90d_60d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
tasks = [
    (
        "BWCN",
        FILE_BWCN_EP,
        os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc"),
    ),
    (
        "B2000WCN",
        FILE_B2000_EP,
        os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"),
    ),
]

for name, ep, o3 in tasks:
    labels, curves = compute_r_curve(name, ep, o3)
    if labels is not None:
        plot_r_curves(name, labels, curves, suffix="ALL")

In [ ]:
# ============================================================
# Block E: Sliding-window correlation curves (LOW 25% years only)
# 目标：只对 low25pct_years_30_70hPa 做 90-day / 60-day 滑动相关曲线
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_EXPERIMENTS = {"B2000WCN", "B2000WCN_NOCOUPL"}
BRIDGE_YEAR = 104

WINDOW_DAYS_LIST = [90, 60]

def build_window_starts():
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def compute_r_curve_low25(exp_name, ep_nc, o3_nc, low25_txt, lat_band=(40, 80), apply_o3_5d=True, min_n=5):
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc) and os.path.exists(low25_txt)):
        print(f"[{exp_name}] Missing file(s).")
        return None, None

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    low25_years = np.loadtxt(low25_txt, dtype=int)
    low25_years = np.atleast_1d(low25_years)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_curves = {wd: [] for wd in WINDOW_DAYS_LIST}

    for window_days in WINDOW_DAYS_LIST:
        for start_month, start_day in starts:
            x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)

            # 连接年修正：仅对拼接型实验，且仅 12 月起点会跨 run 混合
            if (exp_name in BRIDGE_EXPERIMENTS) and (start_month == 12):
                if BRIDGE_YEAR in x_raw.year.values:
                    x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

            common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
            use_years = np.intersect1d(common_years, low25_years)

            x_use = x_raw.sel(year=use_years)
            y_use = y_raw.sel(year=use_years)

            if len(use_years) < min_n:
                r_curves[window_days].append(np.nan)
                continue

            try:
                r, _ = pearsonr(x_use.values, y_use.values)
            except:
                r = np.nan

            r_curves[window_days].append(r)

    ds_ep.close()
    return labels, r_curves

def plot_r_curves(exp_name, labels, r_curves, suffix="LOW25"):
    out_dir = os.path.join(PLOT_BASE, exp_name, "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))

    plt.plot(x, r_curves[90], lw=2.0, marker="o", ms=3, label="90-day window")
    plt.plot(x, r_curves[60], lw=2.0, marker="o", ms=3, label="60-day window")

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels)-1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(f"{exp_name} | Sliding-window correlation with MA minimum O3 ({suffix})",
              loc="left", fontweight="bold")

    plt.legend(frameon=False)
    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_90d_60d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
tasks = [
    (
        "BWCN",
        FILE_BWCN_EP,
        os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN/low25pct_years_30_70hPa.txt")
    ),
    (
        "B2000WCN",
        FILE_B2000_EP,
        os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN/low25pct_years_30_70hPa.txt")
    ),
]

for name, ep, o3, low25 in tasks:
    labels, curves = compute_r_curve_low25(name, ep, o3, low25)
    if labels is not None:
        plot_r_curves(name, labels, curves, suffix="LOW25")

In [ ]:
# ============================================================
# Block D: Sliding-window correlation curves (ALL years, pooled)
# 目标：
# 1) 正确跳过 B2000WCN 的连接年（仅对 12 月起始窗口）
# 2) 将 B2000WCN + BWCN 合并为一个 "interactive ozone long run" 样本
# 3) 90-day 和 60-day 分开作图
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP   = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
DATA_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

def build_window_starts():
    """
    窗口起点：12-01 到 02-01，逐日滑动
    """
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    返回按 year 的 MA minimum O3（目标变量）
    """
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口 Fz 均值
    - 若起点在 12 月：窗口从 (year-1)-12-dd 开始
    - 若起点在 1/2 月：窗口从 year-mm-dd 开始
    """
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_exp_samples_all(exp_name, ep_nc, o3_nc, start_month, start_day, window_days,
                        lat_band=(40, 80), apply_o3_5d=True):
    """
    对单个实验，在给定窗口下返回 DataFrame:
    columns = [exp, year, x_raw, y_raw]
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        print(f"[{exp_name}] Missing file(s).")
        return pd.DataFrame(columns=["exp", "year", "x_raw", "y_raw"])

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    # ✅ 仅对 B2000WCN，在 12 月起始窗口时删除连接年 104
    # 因为这类窗口会混入 Dec(103) + Jan...(104) 的跨 run 人工拼接
    if (exp_name == "B2000WCN") and (start_month == 12):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    x_use = x_raw.sel(year=common_years)
    y_use = y_raw.sel(year=common_years)

    df = pd.DataFrame({
        "exp": exp_name,
        "year": x_use.year.values.astype(int),
        "x_raw": x_use.values.astype(float),
        "y_raw": y_use.values.astype(float),
    })

    ds_ep.close()
    return df

def compute_pooled_r_curve_all(window_days, min_n=8):
    """
    对单个 window_days（90 或 60），计算 pooled interactive ozone long run 的 r 曲线
    """
    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    for start_month, start_day in starts:
        df_bwcn = get_exp_samples_all(
            "BWCN",
            FILE_BWCN_EP,
            os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc"),
            start_month, start_day, window_days
        )
        df_b2000 = get_exp_samples_all(
            "B2000WCN",
            FILE_B2000_EP,
            os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"),
            start_month, start_day, window_days
        )

        df_all = pd.concat([df_bwcn, df_b2000], ignore_index=True).dropna()

        if len(df_all) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(df_all["x_raw"].values, df_all["y_raw"].values)
        except:
            r = np.nan

        r_list.append(r)

    return labels, r_list

def plot_single_curve(labels, r_list, window_days, suffix="ALL"):
    out_dir = os.path.join(PLOT_BASE, "COMBINED_INTERACTIVE", "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))
    plt.plot(x, r_list, lw=2.2, marker="o", ms=3)

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"Interactive Ozone Long Run | {window_days}-day sliding-window correlation with MA minimum O3 ({suffix})",
        loc="left", fontweight="bold"
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_{window_days}d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
for wd in WINDOW_DAYS_LIST:
    labels, r_list = compute_pooled_r_curve_all(window_days=wd)
    plot_single_curve(labels, r_list, window_days=wd, suffix="ALL")

In [ ]:
# ============================================================
# Block E: Sliding-window correlation curves (LOW 25% years, pooled)
# 目标：
# 1) 正确跳过 B2000WCN 的连接年（仅对 12 月起始窗口）
# 2) 将 B2000WCN + BWCN 合并为一个 "interactive ozone long run" 样本
# 3) 仅使用各自 MA O3 minimum 最低 25% 年份
# 4) 90-day 和 60-day 分开作图
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP   = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
DATA_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE      = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

def build_window_starts():
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_exp_samples_low25(exp_name, ep_nc, o3_nc, low25_txt, start_month, start_day, window_days,
                          lat_band=(40, 80), apply_o3_5d=True):
    """
    对单个实验，在给定窗口下返回 low25 年份样本：
    columns = [exp, year, x_raw, y_raw]
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc) and os.path.exists(low25_txt)):
        print(f"[{exp_name}] Missing file(s).")
        return pd.DataFrame(columns=["exp", "year", "x_raw", "y_raw"])

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    low25_years = np.loadtxt(low25_txt, dtype=int)
    low25_years = np.atleast_1d(low25_years)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    # ✅ 仅对 B2000WCN，在 12 月起始窗口时删除连接年 104
    if (exp_name == "B2000WCN") and (start_month == 12):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    use_years = np.intersect1d(common_years, low25_years)

    x_use = x_raw.sel(year=use_years)
    y_use = y_raw.sel(year=use_years)

    df = pd.DataFrame({
        "exp": exp_name,
        "year": x_use.year.values.astype(int),
        "x_raw": x_use.values.astype(float),
        "y_raw": y_use.values.astype(float),
    })

    ds_ep.close()
    return df

def compute_pooled_r_curve_low25(window_days, min_n=5):
    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    for start_month, start_day in starts:
        df_bwcn = get_exp_samples_low25(
            "BWCN",
            FILE_BWCN_EP,
            os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc"),
            os.path.join(DATA_BASE, "BWCN/low25pct_years_30_70hPa.txt"),
            start_month, start_day, window_days
        )
        df_b2000 = get_exp_samples_low25(
            "B2000WCN",
            FILE_B2000_EP,
            os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"),
            os.path.join(DATA_BASE, "B2000WCN/low25pct_years_30_70hPa.txt"),
            start_month, start_day, window_days
        )

        df_all = pd.concat([df_bwcn, df_b2000], ignore_index=True).dropna()

        if len(df_all) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(df_all["x_raw"].values, df_all["y_raw"].values)
        except:
            r = np.nan

        r_list.append(r)

    return labels, r_list

def plot_single_curve(labels, r_list, window_days, suffix="LOW25"):
    out_dir = os.path.join(PLOT_BASE, "COMBINED_INTERACTIVE", "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))
    plt.plot(x, r_list, lw=2.2, marker="o", ms=3)

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"Interactive Ozone Long Run | {window_days}-day sliding-window correlation with MA minimum O3 ({suffix})",
        loc="left", fontweight="bold"
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_{window_days}d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
for wd in WINDOW_DAYS_LIST:
    labels, r_list = compute_pooled_r_curve_low25(window_days=wd)
    plot_single_curve(labels, r_list, window_days=wd, suffix="LOW25")